# sfm_native_adapter_v1

目标：把 `sfm_mse` 从 frequency ridge proxy 推进到 `src/model` 的 native sequence adapter。Notebook 只保留实验流程和中文结论；可复用训练、回测、analysis 逻辑都调用 `src/`。

外部参考只保存 URL 和自有摘要，不复制论文或仓库源码。

In [1]:
from pathlib import Path
import json
import sys

for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / 'src').is_dir() and (candidate / 'AGENTS.md').exists():
        REPO_ROOT = candidate
        break
else:
    raise RuntimeError('repo root not found')

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.data import build_supervised_dataset, load_soxl_support_frame
from src.model.capabilities import model_capability_table, model_fidelity_summary
from src.model.matrix import run_quantbench_matrix_smoke
from src.model.sequence import SFMFrequencyMemoryRegressor
from src.model.training import fit_estimator_for_model, model_adapter_name, model_adapter_status

OUTPUT_DIR = REPO_ROOT / 'research/sfm_native_adapter_v1'
CARD_DIR = OUTPUT_DIR / 'matrix_card'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


## 外部参考摘要

- Hu and Qi, ICML 2017, `State-Frequency Memory Recurrent Neural Networks`：核心思路是把记忆状态显式分解到多个频率分量，帮助序列模型捕捉不同周期模式。参考：https://proceedings.mlr.press/v70/hu17c.html
- stock prediction SFM repo：把 SFM 思路用于多频交易模式发现。参考：https://github.com/z331565360/State-Frequency-Memory-stock-prediction
- 本仓库实现：不迁移 Theano/Keras 旧代码；用自有 PyTorch 小网络保留输入门、Fourier 频率基底、state-frequency memory 和 gated frequency selection。

In [2]:
frame = load_soxl_support_frame(start='2026-01-01', end='2026-06-09')
dataset = build_supervised_dataset(frame, horizon=1)
train = dataset.frame.loc[dataset.frame['date'] <= '2026-04-30'].copy()

estimator, constant_prediction = fit_estimator_for_model(
    'sfm_mse',
    train=train,
    feature_columns=dataset.feature_columns[:5],
)
capability = model_capability_table().loc[lambda df: df['model'].eq('sfm_mse')].iloc[0].to_dict()

assert constant_prediction is None
assert isinstance(estimator, SFMFrequencyMemoryRegressor)
assert hasattr(estimator, 'model_')
assert hasattr(estimator.model_, 'frequency_selector')
assert estimator.lookback_ == 5
assert estimator.feature_count_ == 5
assert estimator.frequency_modes_ == (1, 2, 3)
assert model_adapter_name('sfm_mse') == 'sfm_state_frequency_memory_mse_v1'
assert model_adapter_status('sfm_mse') == 'paper_strict_adapter'
assert capability['paper_fidelity'] == 'native_sequence'
assert capability['is_proxy'] is False

{
    'estimator': type(estimator).__name__,
    'adapter': model_adapter_name('sfm_mse'),
    'frequency_modes': estimator.frequency_modes_,
    'paper_fidelity': capability['paper_fidelity'],
}


{'estimator': 'SFMFrequencyMemoryRegressor',
 'adapter': 'sfm_state_frequency_memory_mse_v1',
 'frequency_modes': (1, 2, 3),
 'paper_fidelity': 'native_sequence'}

In [3]:
model_names = ('ridge_mse', 'gru_mse', 'sfm_mse', 'fedformer_mse')
outputs = run_quantbench_matrix_smoke(
    card_dir=CARD_DIR,
    start='2026-01-01',
    end='2026-06-09',
    train_end='2026-04-30',
    model_names=model_names,
)
comparison = outputs.model_comparison.set_index('model')
fidelity = model_fidelity_summary(model_names)

assert set(outputs.predictions['model'].unique()) == set(model_names)
assert outputs.summary['adapter_statuses'] == ['paper_strict_adapter']
assert outputs.summary['model_fidelity']['proxy_models'] == []
assert 'sfm_mse' in outputs.summary['model_fidelity']['native_models']

comparison.loc[['sfm_mse']]


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


,rank,return_pct,benchmark_return_pct,relative_return_pct,max_drawdown_pct,sharpe,avg_position,turnover,stability_score
model,,,,,,,,,
sfm_mse,3,34.471624,58.118381,-23.646758,25.93482,2.973337,0.719023,4.974616,21.837238


In [4]:
sfm_row = comparison.loc['sfm_mse'].to_dict()
summary = {
    'schema_version': 'research.sfm_native_adapter_v1',
    'status': 'completed',
    'paper_reference': {
        'source_id': 'pmlr:v70:hu17c',
        'url': 'https://proceedings.mlr.press/v70/hu17c.html',
        'usage': '自有摘要和结构假设；不复制论文全文。',
    },
    'stock_reference': {
        'source_id': 'github:z331565360/State-Frequency-Memory-stock-prediction',
        'url': 'https://github.com/z331565360/State-Frequency-Memory-stock-prediction',
        'usage': '用于确认 SFM 在股票多频模式方向的应用背景；不复用源码。',
    },
    'model_names': list(model_names),
    'sfm_estimator': type(estimator).__name__,
    'sfm_adapter': model_adapter_name('sfm_mse'),
    'sfm_frequency_modes': list(estimator.frequency_modes_),
    'sfm_paper_fidelity': capability['paper_fidelity'],
    'sfm_is_proxy': bool(capability['is_proxy']),
    'model_fidelity': fidelity,
    'top_model': str(outputs.summary['top_model']),
    'sfm_return_pct': float(sfm_row['return_pct']),
    'sfm_relative_return_pct': float(sfm_row['relative_return_pct']),
    'sfm_max_drawdown_pct': float(sfm_row['max_drawdown_pct']),
    'sfm_sharpe': float(sfm_row['sharpe']),
    'analysis_summary': outputs.summary['analysis_summary'],
    'artifacts': {
        'matrix_card': 'matrix_card/',
        'analysis_summary': 'matrix_card/analysis_summary.json',
        'model_comparison': 'matrix_card/model_comparison.csv',
        'model_capabilities': 'matrix_card/model_capabilities.csv',
        'daily_backtest': 'matrix_card/daily_backtest.csv',
        'model_run_ledger': 'matrix_card/model_run_ledger.csv',
        'model_run_ledger_summary': 'matrix_card/model_run_ledger_summary.json',
        'artifact_manifest': 'matrix_card/artifact_manifest.json',
    },
}
(OUTPUT_DIR / 'notebook_summary.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
summary


{'schema_version': 'research.sfm_native_adapter_v1',
 'status': 'completed',
 'paper_reference': {'source_id': 'pmlr:v70:hu17c',
  'url': 'https://proceedings.mlr.press/v70/hu17c.html',
  'usage': '自有摘要和结构假设；不复制论文全文。'},
 'stock_reference': {'source_id': 'github:z331565360/State-Frequency-Memory-stock-prediction',
  'url': 'https://github.com/z331565360/State-Frequency-Memory-stock-prediction',
  'usage': '用于确认 SFM 在股票多频模式方向的应用背景；不复用源码。'},
 'model_names': ['ridge_mse', 'gru_mse', 'sfm_mse', 'fedformer_mse'],
 'sfm_estimator': 'SFMFrequencyMemoryRegressor',
 'sfm_adapter': 'sfm_state_frequency_memory_mse_v1',
 'sfm_frequency_modes': [1, 2, 3],
 'sfm_paper_fidelity': 'native_sequence',
 'sfm_is_proxy': False,
 'model_fidelity': {'schema_version': 'model.fidelity.v1',
  'model_count': 4,
  'fidelity_counts': {'native_library': 1,
   'native_sequence': 2,
   'native_torch': 1},
  'proxy_models': [],
  'native_models': ['fedformer_mse', 'gru_mse', 'ridge_mse', 'sfm_mse'],
  'promotion_blocke

## 中文结论

- 做了什么：在 `src/model.sequence` 增加 `SFMFrequencyMemoryRegressor`，用 PyTorch 小网络实现输入门、Fourier 频率基底、state-frequency memory 和 gated frequency selection；`sfm_mse` 训练路径已从 frequency ridge proxy 切到该 estimator，adapter 名称为 `sfm_state_frequency_memory_mse_v1`。
- 核心指标：本 notebook 训练并回测 `ridge_mse`、`gru_mse`、`sfm_mse`、`fedformer_mse`，输出 `model_comparison.csv`、`daily_backtest.csv`、`model_run_ledger.csv`、`qlib_risk_metrics.csv`、`model_effect_deltas.csv` 和 manifest。具体收益、回撤、Sharpe 以 `notebook_summary.json` 与 `matrix_card/analysis_summary.json` 为准。
- regime 差异：SFM 的价值不应只看全区间收益；重点看频率记忆在 downtrend/stress/window_regime 中是否比 GRU/FEDformer 更稳。
- 是否晋升：否。`native_sequence` 只表示实现形态已从 proxy 推进为稳定训练接口；production 晋升仍需要更长历史窗口、更多市场 regime、成本压力和 paper/sim replay 复验。
- 风险：这是 compact 自有实现，不是逐行复刻论文/Theano 版本；后续如果需要更高 paper fidelity，应继续加入 joint state-frequency forget gate、frequency-specific output gate 和 adaptive frequency。
- 下一步：继续按 backlog 验证 graph/temporal graph proxy replacement，例如 `market_graph_mse`、`gcn_mse`、`rsr_mse`、`thgnn_mse`。